In [ ]:
import sys
import sklearn
import joblib

print("Versión de Python:", sys.version)
print("Versión de scikit-learn:", sklearn.__version__)
print("Versión de joblib:", joblib.__version__)

Versión de Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Versión de scikit-learn: 1.6.1
Versión de joblib: 1.6.0


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Actividad6_Customer_Churn_ENTREGA.zip to Actividad6_Customer_Churn_ENTREGA.zip


In [ ]:
import zipfile

zip_path = next(iter(uploaded.keys()))

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    archivos = zip_ref.namelist()

    modelos = [
        archivo for archivo in archivos
        if archivo.endswith(".joblib")
    ]

    print("Modelos encontrados:")
    for modelo in modelos:
        print(modelo)

Modelos encontrados:
Actividad6_Customer_Churn/evidencias/modelo_LogisticRegression.joblib
Actividad6_Customer_Churn/evidencias/modelo_RandomForest.joblib


In [ ]:
import zipfile
import os

carpeta_modelos = "/content/modelos_churn"
os.makedirs(carpeta_modelos, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    for archivo in modelos:
        zip_ref.extract(archivo, carpeta_modelos)

print("Modelos extraídos correctamente.")

Modelos extraídos correctamente.


In [ ]:
import joblib
import os

for archivo in modelos:
    ruta = os.path.join(carpeta_modelos, archivo)

    print(f"\nCargando: {os.path.basename(archivo)}")

    try:
        modelo = joblib.load(ruta)
        print("Carga correcta:", type(modelo))
    except Exception as e:
        print("Error al cargar:", repr(e))


Cargando: modelo_LogisticRegression.joblib
Carga correcta: <class 'sklearn.pipeline.Pipeline'>

Cargando: modelo_RandomForest.joblib
Carga correcta: <class 'sklearn.pipeline.Pipeline'>


In [ ]:
import joblib

modelo_lr = joblib.load(
    "/content/modelos_churn/Actividad6_Customer_Churn/evidencias/modelo_LogisticRegression.joblib"
)

modelo_rf = joblib.load(
    "/content/modelos_churn/Actividad6_Customer_Churn/evidencias/modelo_RandomForest.joblib"
)

print("Columnas esperadas por Logistic Regression:")
print(modelo_lr.feature_names_in_)

print("\nColumnas esperadas por Random Forest:")
print(modelo_rf.feature_names_in_)

Columnas esperadas por Logistic Regression:
['gender' 'SeniorCitizen' 'Partner' 'Dependents' 'tenure' 'PhoneService'
 'MultipleLines' 'InternetService' 'OnlineSecurity' 'OnlineBackup'
 'DeviceProtection' 'TechSupport' 'StreamingTV' 'StreamingMovies'
 'Contract' 'PaperlessBilling' 'PaymentMethod' 'MonthlyCharges'
 'TotalCharges']

Columnas esperadas por Random Forest:
['gender' 'SeniorCitizen' 'Partner' 'Dependents' 'tenure' 'PhoneService'
 'MultipleLines' 'InternetService' 'OnlineSecurity' 'OnlineBackup'
 'DeviceProtection' 'TechSupport' 'StreamingTV' 'StreamingMovies'
 'Contract' 'PaperlessBilling' 'PaymentMethod' 'MonthlyCharges'
 'TotalCharges']


In [ ]:
import pandas as pd

cliente_prueba = pd.DataFrame([{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "DSL",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 50.0,
    "TotalCharges": 600.0
}])

print(cliente_prueba)

   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No      12          Yes   

  MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection  \
0            No             DSL            Yes           No               No   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No          No              No  Month-to-month              Yes   

      PaymentMethod  MonthlyCharges  TotalCharges  
0  Electronic check            50.0         600.0  


In [ ]:
print("Logistic Regression:")
print("Predicción:", modelo_lr.predict(cliente_prueba))
print("Probabilidades:", modelo_lr.predict_proba(cliente_prueba))

print("\nRandom Forest:")
print("Predicción:", modelo_rf.predict(cliente_prueba))
print("Probabilidades:", modelo_rf.predict_proba(cliente_prueba))

Logistic Regression:
Predicción: [1]
Probabilidades: [[0.44198872 0.55801128]]

Random Forest:
Predicción: [0]
Probabilidades: [[0.57675432 0.42324568]]


In [ ]:
print("Clases Logistic Regression:")
print(modelo_lr.classes_)

print("\nClases Random Forest:")
print(modelo_rf.classes_)

Clases Logistic Regression:
[0 1]

Clases Random Forest:
[0 1]


In [ ]:
!pip install fastapi uvicorn

In [ ]:
%%writefile app.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib

# Cargar modelos
modelo_lr = joblib.load(
    "/content/modelos_churn/Actividad6_Customer_Churn/evidencias/modelo_LogisticRegression.joblib"
)

modelo_rf = joblib.load(
    "/content/modelos_churn/Actividad6_Customer_Churn/evidencias/modelo_RandomForest.joblib"
)

app = FastAPI(
    title="Customer Churn API",
    description="API para predecir la posible cancelación de clientes.",
    version="1.0.0"
)


# Datos que recibirá la API
class Cliente(BaseModel):
    gender: str
    SeniorCitizen: int
    Partner: str
    Dependents: str
    tenure: int
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str
    MonthlyCharges: float
    TotalCharges: float


@app.get("/health")
def health():
    return {"status": "ok", "message": "Customer Churn API activa"}


@app.post("/predict")
def predict(cliente: Cliente, modelo: str = "logistic_regression"):

    # Seleccionar modelo
    if modelo == "logistic_regression":
        model = modelo_lr
    elif modelo == "random_forest":
        model = modelo_rf
    else:
        raise HTTPException(
            status_code=400,
            detail="Modelo no válido. Usa logistic_regression o random_forest."
        )

    # Convertir datos a DataFrame
    datos = pd.DataFrame([cliente.model_dump()])

    # Obtener predicción y probabilidades
    prediccion = int(model.predict(datos)[0])
    probabilidades = model.predict_proba(datos)[0]

    return {
        "modelo": modelo,
        "prediccion": prediccion,
        "resultado": (
            "Posible churn" if prediccion == 1 else "Cliente permanece"
        ),
        "probabilidad_no_churn": float(probabilidades[0]),
        "probabilidad_churn": float(probabilidades[1])
    }

Writing app.py


In [ ]:
from app import app

print("API importada correctamente")
print("Rutas disponibles:")

for ruta in app.routes:
    print(ruta.path, ruta.methods)

API importada correctamente
Rutas disponibles:
/openapi.json {'GET', 'HEAD'}
/docs {'GET', 'HEAD'}
/docs/oauth2-redirect {'GET', 'HEAD'}
/redoc {'GET', 'HEAD'}
/health {'GET'}
/predict {'POST'}


In [ ]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

# Prueba de salud
respuesta_health = client.get("/health")

print("Health:")
print(respuesta_health.status_code)
print(respuesta_health.json())

Health:
200
{'status': 'ok', 'message': 'Customer Churn API activa'}


In [ ]:
datos_cliente = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "DSL",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 50.0,
    "TotalCharges": 600.0
}

# Logistic Regression
respuesta_lr = client.post(
    "/predict?modelo=logistic_regression",
    json=datos_cliente
)

print("Logistic Regression:")
print(respuesta_lr.status_code)
print(respuesta_lr.json())

# Random Forest
respuesta_rf = client.post(
    "/predict?modelo=random_forest",
    json=datos_cliente
)

print("\nRandom Forest:")
print(respuesta_rf.status_code)
print(respuesta_rf.json())

Logistic Regression:
200
{'modelo': 'logistic_regression', 'prediccion': 1, 'resultado': 'Posible churn', 'probabilidad_no_churn': 0.44198872161622427, 'probabilidad_churn': 0.5580112783837757}

Random Forest:
200
{'modelo': 'random_forest', 'prediccion': 0, 'resultado': 'Cliente permanece', 'probabilidad_no_churn': 0.5767543243967318, 'probabilidad_churn': 0.4232456756032682}


In [ ]:
respuesta_error = client.post(
    "/predict?modelo=modelo_inventado",
    json=datos_cliente
)

print("Código:", respuesta_error.status_code)
print("Respuesta:", respuesta_error.json())

Código: 400
Respuesta: {'detail': 'Modelo no válido. Usa logistic_regression o random_forest.'}


In [ ]:
import os

ruta_modelos = "/content/modelos_churn/Actividad6_Customer_Churn/evidencias"

print("Archivos encontrados:")
print(os.listdir(ruta_modelos))

Archivos encontrados:
['modelo_LogisticRegression.joblib', 'modelo_RandomForest.joblib']


In [ ]:
import os
import shutil

proyecto = "/content/customer-churn-api"
carpeta_modelos = os.path.join(proyecto, "modelos")

os.makedirs(carpeta_modelos, exist_ok=True)

# Copiar la API
shutil.copy("/content/app.py", os.path.join(proyecto, "app.py"))

# Copiar los modelos
shutil.copy(
    "/content/modelos_churn/Actividad6_Customer_Churn/evidencias/modelo_LogisticRegression.joblib",
    os.path.join(carpeta_modelos, "modelo_LogisticRegression.joblib")
)

shutil.copy(
    "/content/modelos_churn/Actividad6_Customer_Churn/evidencias/modelo_RandomForest.joblib",
    os.path.join(carpeta_modelos, "modelo_RandomForest.joblib")
)

print("Proyecto preparado:")
for raiz, carpetas, archivos in os.walk(proyecto):
    print(raiz, archivos)

Proyecto preparado:
/content/customer-churn-api ['app.py']
/content/customer-churn-api/modelos ['modelo_LogisticRegression.joblib', 'modelo_RandomForest.joblib']


In [ ]:
%%writefile /content/customer-churn-api/app.py

from pathlib import Path

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib

# Ruta de la carpeta del proyecto
BASE_DIR = Path(__file__).resolve().parent
MODELS_DIR = BASE_DIR / "modelos"

# Cargar modelos
modelo_lr = joblib.load(
    MODELS_DIR / "modelo_LogisticRegression.joblib"
)

modelo_rf = joblib.load(
    MODELS_DIR / "modelo_RandomForest.joblib"
)

app = FastAPI(
    title="Customer Churn API",
    description="API para predecir la posible cancelación de clientes.",
    version="1.0.0"
)


class Cliente(BaseModel):
    gender: str
    SeniorCitizen: int
    Partner: str
    Dependents: str
    tenure: int
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str
    MonthlyCharges: float
    TotalCharges: float


@app.get("/health")
def health():
    return {
        "status": "ok",
        "message": "Customer Churn API activa"
    }


@app.post("/predict")
def predict(cliente: Cliente, modelo: str = "logistic_regression"):

    if modelo == "logistic_regression":
        model = modelo_lr
    elif modelo == "random_forest":
        model = modelo_rf
    else:
        raise HTTPException(
            status_code=400,
            detail="Modelo no válido. Usa logistic_regression o random_forest."
        )

    datos = pd.DataFrame([cliente.model_dump()])

    prediccion = int(model.predict(datos)[0])
    probabilidades = model.predict_proba(datos)[0]

    return {
        "modelo": modelo,
        "prediccion": prediccion,
        "resultado": (
            "Posible churn" if prediccion == 1 else "Cliente permanece"
        ),
        "probabilidad_no_churn": float(probabilidades[0]),
        "probabilidad_churn": float(probabilidades[1])
    }

Overwriting /content/customer-churn-api/app.py


In [ ]:
%%writefile /content/customer-churn-api/requirements.txt

fastapi
uvicorn
pandas
scikit-learn==1.6.1
joblib

Overwriting /content/customer-churn-api/requirements.txt


In [ ]:
import sys

sys.path.insert(0, "/content/customer-churn-api")

from app import app
from fastapi.testclient import TestClient

client = TestClient(app)

respuesta = client.get("/health")

print("Código:", respuesta.status_code)
print("Respuesta:", respuesta.json())

Código: 200
Respuesta: {'status': 'ok', 'message': 'Customer Churn API activa'}


In [ ]:
%%writefile /content/customer-churn-api/Dockerfile

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY modelos/ ./modelos/

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing /content/customer-churn-api/Dockerfile


In [ ]:
%%writefile /content/customer-churn-api/.dockerignore

__pycache__/
*.pyc
.ipynb_checkpoints/
.env

Writing /content/customer-churn-api/.dockerignore


In [ ]:
import os

proyecto = "/content/customer-churn-api"

print("Archivos del proyecto:")
for raiz, carpetas, archivos in os.walk(proyecto):
    for archivo in archivos:
        print(os.path.join(raiz, archivo))

Archivos del proyecto:
/content/customer-churn-api/.dockerignore
/content/customer-churn-api/Dockerfile
/content/customer-churn-api/requirements.txt
/content/customer-churn-api/app.py
/content/customer-churn-api/modelos/modelo_LogisticRegression.joblib
/content/customer-churn-api/modelos/modelo_RandomForest.joblib


In [ ]:
import shutil

ruta_zip = shutil.make_archive(
    "/content/customer-churn-api",
    "zip",
    "/content/customer-churn-api"
)

print("ZIP creado:", ruta_zip)

ZIP creado: /content/customer-churn-api.zip


In [ ]:
from google.colab import files

files.download("/content/customer-churn-api.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>